# Assignment: Extend the az.ipynb Lab

**Based on:** `Lab2.ipynb` (the Module 3 lab).

This week's assignment is short on purpose: take your working `Lab2.ipynb` lab and add **one
more step** to the chain. No new concepts, no new setup, no new libraries — just one more
chained LLM call that builds on what you already have.

**Two deployments this time:** `gpt-5.1-ptu` is the default deployment for every existing
step (Steps 1–4). The new step you add (Step 5) must call `gpt-5.4-ptu` instead.

## Step 1 — Start from your working lab

- Make a copy of your completed `Lab2.ipynb` (e.g. rename the copy `assignment3.ipynb`), or
  continue directly inside this notebook — either is fine.
- Copy in your working code from the lab's Steps 1–4: the imports and `.env` config, the
  `AzureOpenAI` client, the `chat()` helper, and the chain itself (fun fact → generate a hard
  question → answer it → evaluate the answer).
- Confirm your `.env`'s `AZURE_APIM_OPENAI_DEPLOYMENT` is set to `gpt-5.1-ptu` — this stays
  the default deployment for Steps 1–4, unchanged.

Run those cells first and confirm they still work before moving on.

In [1]:
# TODO: paste your working Lab2.ipynb code here (Steps 1-4):
from dotenv import load_dotenv
import os
import sys

from openai import AzureOpenAI, OpenAI
api_key = os.getenv("AZURE_APIM_OPENAI_SUBSCRIPTION_KEY")
api_version = os.getenv("AZURE_APIM_OPENAI_API_VERSION")
endpoint = os.getenv("AZURE_APIM_OPENAI_ENDPOINT")
deployment = os.getenv("AZURE_APIM_OPENAI_DEPLOYMENT")
if not all([api_key, api_version, endpoint, deployment]):
    sys.exit(
        "Missing Azure APIM settings. Set AZURE_APIM_OPENAI_SUBSCRIPTION_KEY, "
        "AZURE_APIM_OPENAI_API_VERSION, AZURE_APIM_OPENAI_ENDPOINT, and "
        "AZURE_APIM_OPENAI_DEPLOYMENT in your .env file."
    )

print(f"Azure APIM key exists and begins {api_key[:8]}")
print(f"Deployment: {deployment}")
openai = AzureOpenAI(
    api_key=api_key,
    api_version=api_version,
    azure_endpoint=endpoint,
)
def chat(messages, max_completion_tokens=5000, deployment=deployment):
    return openai.chat.completions.create(
    model=deployment, messages=messages,
    max_completion_tokens=max_completion_tokens,
    )
messages = [{"role": "user", "content": "Tell me a short fun fact"}]
response = chat(messages)
print(response.choices[0].message.content)
question = (
    "Please propose a hard, challenging question to assess someone's IQ. "
    "Respond only with the question."
)
messages = [{"role": "user", "content": question}]
response = chat(messages)
question = response.choices[0].message.content
print(question)
messages = [{"role": "user", "content": question}]
response = chat(messages)
answer = response.choices[0].message.content
print(answer)
deployment="gpt-5.4-ptu"
message = f"""
Here is a question:
{question}

And here is a possible answer that might be correct or incorrect:
{answer}

Please evaluate if the answer is correct or incorrect.
"""
print(message)



Azure APIM key exists and begins 233a43f0
Deployment: gpt-5.1-ptu
Octopuses have three hearts and blue blood—and when they’re stressed, they can squirt ink that actually dulls a predator’s sense of smell.
A train leaves City A for City B, 300 km away, at 9:00 a.m. and travels at a constant but unknown speed. At 10:00 a.m., a second train leaves City B for City A on the same track, traveling 25% faster than the first train. At 10:30 a.m., a bird starts flying from the first train toward the second train at a constant speed of 60 km/h. Every time the bird reaches a train, it instantly turns around and flies toward the other one, continuing this back-and-forth motion until the trains meet. The trains pass each other exactly at 11:00 a.m. What total distance does the bird travel, and what is the speed of the first train?
Key is to use the meeting time of the trains and the bird’s flight duration.

**1. Determine the speed of the first train**

Let the speed of the first train be \(v\) km/h

## Step 2 — Add one more chained step

Add a **5th step** to the chain. Its prompt must be built from at least one variable you
already have (`fact`, `question`, `answer`, or the evaluation text) — the same chaining
pattern as every other step in the lab.

**This step must call `gpt-5.4-ptu`, not the default deployment.** Pass it explicitly when
you call `chat()`:

```python
response = chat(messages, deployment="gpt-5.4-ptu")
```

Pick **one** idea below, or invent your own:

- Rate the difficulty of the question on a 1–10 scale, with a one-sentence justification.
- Rewrite the answer in one simple sentence a 10-year-old could understand.
- Suggest one new, related fun fact that connects to the original topic.
- Translate the final answer into a language of your choice.
- Write a one-line verdict on whether the model's own answer was actually correct, and why.

Store the result in its own variable, and print it clearly labeled (e.g. `=== STEP 5
(gpt-5.4-ptu) ===`).

In [2]:
# TODO: Step 5 - build a new prompt using an earlier variable
#   (fact, question, answer, and/or the evaluation)
# TODO: call chat(messages, deployment="gpt-5.4-ptu") -- do NOT use the default deployment here
# TODO: extract the result, and print it clearly labeled
deployment="gpt-5.4-ptu"
step5_prompt = f"""
Rewrite the following answer to this question in one simple sentence that a 10-year-old could easily understand.

{question, answer}
"""

messages = [{"role": "user", "content": step5_prompt}]
response = chat(messages, deployment="gpt-5.4-ptu")
step5_result = response.choices[0].message.content

print("=== STEP 5 (gpt-5.4-ptu) ===")
print(step5_result)



=== STEP 5 (gpt-5.4-ptu) ===
The first train goes about 92.3 km/h, and because the bird flies for 30 minutes at 60 km/h until the trains meet, it travels 30 km total.


## Reflection

Answer in a sentence or two each:

1. **Which earlier variable(s) did your Step 5 prompt use, and why that one?**
2. **What would break if you ran Step 5 before the step it depends on?**
3. **Why might a real project deliberately use a different deployment (e.g. a stronger or
   more expensive model) for just one step in a chain, instead of using it everywhere?**

### My reflection

1. I used the question and answer variables in order to give the llm  the full context of the situation
2. If you ran step 5 without the prior steps, the llm wouldn't have anything to go on, so there would be an error with undefined values
3. Cost-saving measures in order to avoid utilizing high-cost llms for low-cost problems.

## Submission checklist

- [ ] Notebook runs top to bottom without errors (`Kernel → Restart & Run All`)
- [ ] `.env` file is **not** included in your submission
- [ ] Step 5 is clearly labeled and its prompt uses at least one earlier variable
- [ ] Reflection questions are answered